# Main Figure v2 — Annotation Concordance (Multi-Panel)

Full-page figure showing agreement between Ensembl and CAT annotations across
462 human genome assemblies. Progressive story:

- **Panel A**: Gene-level agreement (~97.5%) — 3 style prototypes to choose from
- **Panel B**: Transcript concordance by biotype with intron-chain resolution
- **Panel C**: Jaccard index distribution by biotype (continuous overlap)
- **Panel D**: CDS concordance for protein-coding genes (~99.8%)
- **Panel E**: Per-assembly divergence category summary

### Data sources
- Gene presence: `qc_metrics/*/gene_presence.tsv` or `summary_stats/funnel_rung1_gene_presence_per_asm.tsv`
- Intron chain + Jaccard: `intermediate_spreadsheets/intron_chain/` (from `aggregate_intron_chain_by_biotype.py`)
- CDS integrity: `intermediate_spreadsheets/coding_integrity/`
- GRCh38 divergence: `intermediate_spreadsheets/divergence/`

In [ ]:
import os, shutil
from pathlib import Path
import matplotlib as mpl
from matplotlib import font_manager as fm

# Use local scratch for Matplotlib cache to avoid stale NFS handles
try:
    MPLDIR = Path('/tmp') / f"{os.environ.get('USER','user')}-mplconfig"
    MPLDIR.mkdir(parents=True, exist_ok=True)
    os.environ['MPLCONFIGDIR'] = str(MPLDIR)
    import matplotlib
    ttf_src = Path(matplotlib.get_data_path())/'fonts'/'ttf'
    ttf_dst = MPLDIR/'ttf'
    shutil.copytree(ttf_src, ttf_dst, dirs_exist_ok=True)
    for f in ttf_dst.glob('*.ttf'):
        fm.fontManager.addfont(str(f))
    fm._load_fontmanager(try_read_cache=False)
    mpl.rcParams.update({'svg.fonttype':'none', 'pdf.fonttype':42, 'ps.fonttype':42,
                         'font.family':'DejaVu Sans'})
except Exception as e:
    print('Font init warning:', e)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import matplotlib.patches as mpatches
import warnings
import gc

warnings.filterwarnings('ignore')

# Publication-quality defaults
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 8,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 7,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

print(f'pandas {pd.__version__}, numpy {np.__version__}')

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
OUTPUT_DIR  = Path('/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results')

QC_DIR         = OUTPUT_DIR / 'qc_metrics'
RESULTS_DIR    = OUTPUT_DIR / 'results'
SUMMARY_DIR    = OUTPUT_DIR / 'summary_stats'
FIGURE_DIR     = OUTPUT_DIR / 'figures'
INTRON_DIR     = OUTPUT_DIR / 'intermediate_spreadsheets' / 'intron_chain'
CDS_DIR        = OUTPUT_DIR / 'intermediate_spreadsheets' / 'coding_integrity'
DIV_DIR        = OUTPUT_DIR / 'intermediate_spreadsheets' / 'divergence'
FIGURE_DIR.mkdir(exist_ok=True, parents=True)

print(f'Output:  {OUTPUT_DIR}')
print(f'Figures: {FIGURE_DIR}')

## Colour palette

The intron-chain classification uses a colour scheme where visually similar
shades group into 3 summary categories:

| Group | Classes | Colour family |
|-------|---------|---------------|
| **Exact match** | `Exact_Match` | Green |
| **Same intron chain** | `Intron_Match`, `Intron_Subset`, `Intron_Superset` | Blues |
| **Partial overlap** | `Partial_5`, `Partial_3`, `Other_Partial` | Oranges |
| **No match** | `No_Match` | Red |

In [ ]:
# ── Classification colours (8-class detail, visually grouped) ──────────────
CLASS_COLORS = {
    'Exact_Match':    '#2a9d8f',  # Teal-green
    'Intron_Match':   '#264653',  # Dark navy-blue
    'Intron_Subset':  '#457b9d',  # Steel blue
    'Intron_Superset':'#74a9cf',  # Light blue
    'Partial_5':      '#e9c46a',  # Gold
    'Partial_3':      '#f4a261',  # Sandy orange
    'Other_Partial':  '#e76f51',  # Burnt orange
    'No_Match':       '#c1121f',  # Red
}

CLASS_LABELS = {
    'Exact_Match':     'Exact match',
    'Intron_Match':    'Intron chain match',
    'Intron_Subset':   'Intron subset',
    'Intron_Superset': 'Intron superset',
    'Partial_5':       "Partial (5')",
    'Partial_3':       "Partial (3')",
    'Other_Partial':   'Other partial',
    'No_Match':        'No match',
}

CLASSIFICATION_ORDER = [
    'Exact_Match', 'Intron_Match', 'Intron_Subset', 'Intron_Superset',
    'Partial_5', 'Partial_3', 'Other_Partial', 'No_Match',
]

# 3-group summary mapping
GROUP_3_MAP = {
    'Exact_Match':    'Exact match',
    'Intron_Match':   'Same intron chain',
    'Intron_Subset':  'Same intron chain',
    'Intron_Superset':'Same intron chain',
    'Partial_5':      'Partial/None',
    'Partial_3':      'Partial/None',
    'Other_Partial':  'Partial/None',
    'No_Match':       'Partial/None',
}

# 4-group summary mapping
GROUP_4_MAP = {
    'Exact_Match':    'Exact match',
    'Intron_Match':   'Same intron chain',
    'Intron_Subset':  'Same intron chain',
    'Intron_Superset':'Same intron chain',
    'Partial_5':      'Partial overlap',
    'Partial_3':      'Partial overlap',
    'Other_Partial':  'Partial overlap',
    'No_Match':       'No match',
}

GROUP_4_COLORS = {
    'Exact match':       '#2a9d8f',
    'Same intron chain': '#457b9d',
    'Partial overlap':   '#f4a261',
    'No match':          '#c1121f',
}
GROUP_4_ORDER = ['Exact match', 'Same intron chain', 'Partial overlap', 'No match']

BIOTYPE_ORDER  = ['protein_coding', 'lncRNA', 'pseudogene', 'other_ncRNA', 'other']
BIOTYPE_LABELS = {
    'protein_coding': 'Protein-coding',
    'lncRNA':         'lncRNA',
    'pseudogene':     'Pseudogene',
    'other_ncRNA':    'Other ncRNA',
    'other':          'Other',
}

---
## Load data

In [ ]:
# ── Panel A data: Gene-level presence per assembly ─────────────────────────
# Try pre-computed funnel summary first; fall back to recomputing
funnel_file = SUMMARY_DIR / 'funnel_rung1_gene_presence_per_asm.tsv'
if funnel_file.exists():
    gene_pres = pd.read_csv(funnel_file, sep='\t')
    print(f'Loaded gene presence: {len(gene_pres)} assemblies')
else:
    # Recompute from raw gene presence files
    import re
    ACC_RE = re.compile(r'(GC[AF]_\d+\.\d+)')
    files = sorted(QC_DIR.rglob('*_gene_presence.tsv'))
    print(f'Computing gene presence from {len(files)} files...')
    rows = []
    for f in files:
        m = ACC_RE.search(f.name)
        acc = m.group(1) if m else f.stem
        df = pd.read_csv(f, sep='\t')
        for col in ['present_in_ensembl', 'present_in_cat']:
            df[col] = df[col].map({'True': True, 'False': False, True: True, False: False})
        df = df[~df['gene_name'].str.match(r'^ENSG', na=False)]
        n_union = len(df)
        n_both = ((df['present_in_ensembl']) & (df['present_in_cat'])).sum()
        rows.append({'assembly_accession': acc,
                     'n_union_loci': n_union, 'n_both_loci': int(n_both),
                     'pct_gene_presence': n_both / n_union if n_union > 0 else np.nan})
    gene_pres = pd.DataFrame(rows)
    print(f'Computed gene presence: {len(gene_pres)} assemblies')

gene_pres['pct'] = gene_pres['pct_gene_presence'] * 100
print(f'  Median: {gene_pres["pct"].median():.1f}%  '
      f'IQR: {gene_pres["pct"].quantile(0.25):.1f}–{gene_pres["pct"].quantile(0.75):.1f}%')

In [ ]:
# ── Panel B data: Intron chain classification by biotype ──────────────────
ic_file = INTRON_DIR / 'intron_chain_by_biotype_per_assembly.tsv'
if not ic_file.exists():
    print(f'WARNING: {ic_file} not found.')
    print('Run:  python bin/aggregate_intron_chain_by_biotype.py \\')
    print('        --transcript-concordance-dir <qc_metrics_dir> \\')
    print('        --output-dir <output_dir>/intermediate_spreadsheets/intron_chain')
    ic_data = pd.DataFrame()
else:
    ic_data = pd.read_csv(ic_file, sep='\t')
    print(f'Loaded intron chain data: {len(ic_data):,} rows, '
          f'{ic_data["assembly_accession"].nunique()} assemblies')

# ── Panel C data: Jaccard by biotype ─────────────────────────────────────
jac_file = INTRON_DIR / 'jaccard_by_biotype_per_assembly.tsv'
if not jac_file.exists():
    print(f'WARNING: {jac_file} not found.')
    jac_data = pd.DataFrame()
else:
    jac_data = pd.read_csv(jac_file, sep='\t')
    print(f'Loaded Jaccard data: {len(jac_data):,} rows')

In [ ]:
# ── Panel D data: CDS concordance ────────────────────────────────────────
cds_file = CDS_DIR / 'cds_classification_distribution.tsv'
cds_asm_file = CDS_DIR / 'coding_integrity_per_assembly.tsv'
cds_dist = pd.read_csv(cds_file, sep='\t') if cds_file.exists() else pd.DataFrame()
cds_asm = pd.read_csv(cds_asm_file, sep='\t') if cds_asm_file.exists() else pd.DataFrame()
if not cds_dist.empty:
    print(f'CDS classification distribution: {len(cds_dist)} rows')
if not cds_asm.empty:
    print(f'CDS per-assembly: {len(cds_asm)} assemblies')

In [ ]:
# ── Panel E data: GRCh38 divergence ──────────────────────────────────────
div_file = DIV_DIR / 'grch38_divergence_cross_tab.tsv'
div_asm_file = DIV_DIR / 'grch38_divergence_per_assembly.tsv'
div_cross = pd.read_csv(div_file, sep='\t') if div_file.exists() else pd.DataFrame()
div_asm = pd.read_csv(div_asm_file, sep='\t') if div_asm_file.exists() else pd.DataFrame()
if not div_cross.empty:
    print(f'Divergence cross-tab: {len(div_cross)} categories')
if not div_asm.empty:
    print(f'Divergence per-assembly: {div_asm["assembly_accession"].nunique()} assemblies')

---
## Panel A — Gene-level agreement (3 style prototypes)

All three use the same data: per-assembly % of gene loci detected by both methods.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
vals = gene_pres['pct'].dropna()
med = vals.median()
q25, q75 = vals.quantile(0.25), vals.quantile(0.75)

# ── A1: Jittered strip + boxplot ─────────────────────────────────────────
ax = axes[0]
bp = ax.boxplot(vals, vert=True, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor='#2a9d8f', alpha=0.3),
                medianprops=dict(color='#264653', linewidth=2),
                whiskerprops=dict(color='#264653'),
                capprops=dict(color='#264653'),
                flierprops=dict(marker='o', markersize=3, alpha=0.5))
jitter = np.random.default_rng(42).uniform(-0.15, 0.15, len(vals))
ax.scatter(np.ones(len(vals)) + jitter, vals, s=6, alpha=0.35,
           color='#2a9d8f', edgecolors='none', zorder=3)
ax.set_ylabel('Gene loci detected by both methods (%)')
ax.set_title(f'A1. Strip + Boxplot\nMedian: {med:.1f}%', fontsize=9)
ax.set_xticks([1])
ax.set_xticklabels(['All assemblies'])
ax.set_ylim(max(vals.min() - 1, 90), 100)

# ── A2: Horizontal median bar + dot strip (ladder-style) ─────────────────
ax = axes[1]
ax.barh(0, med, height=0.5, color='#2a9d8f', alpha=0.85, edgecolor='white')
ax.text(med + 0.1, 0, f'{med:.1f}%', va='center', fontsize=9, fontweight='bold')
jitter_y = np.random.default_rng(42).uniform(-0.18, 0.18, len(vals))
ax.scatter(vals, jitter_y, s=6, alpha=0.4, color='#264653', edgecolors='none', zorder=3)
ax.set_xlabel('Gene loci detected by both methods (%)')
ax.set_title(f'A2. Horizontal bar + strip\nMedian: {med:.1f}%', fontsize=9)
ax.set_yticks([])
ax.set_xlim(max(vals.min() - 1, 90), 101)
ax.axvline(med, color='#264653', linewidth=0.8, linestyle='--', alpha=0.5)

# ── A3: Violin plot ──────────────────────────────────────────────────────
ax = axes[2]
parts = ax.violinplot(vals, vert=True, showmedians=True, showextrema=True)
for pc in parts['bodies']:
    pc.set_facecolor('#2a9d8f')
    pc.set_alpha(0.5)
parts['cmedians'].set_color('#264653')
parts['cmedians'].set_linewidth(2)
ax.set_ylabel('Gene loci detected by both methods (%)')
ax.set_title(f'A3. Violin\nMedian: {med:.1f}%', fontsize=9)
ax.set_xticks([1])
ax.set_xticklabels(['All assemblies'])
ax.set_ylim(max(vals.min() - 1, 90), 100)

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(FIGURE_DIR / 'panel_A_prototypes.png', dpi=300, bbox_inches='tight')
fig.savefig(FIGURE_DIR / 'panel_A_prototypes.pdf', bbox_inches='tight')
plt.show()
print(f'Saved panel_A_prototypes.{{png,pdf}}')

---
## Panel B — Transcript concordance by biotype

Stacked horizontal bars showing gene-level best-match intron-chain
classification per biotype. Two versions:
- **B1**: Full 8-class resolution (visually grouped by colour family)
- **B2**: Collapsed 4-group summary

In [ ]:
if ic_data.empty:
    print('Skipping Panel B — no intron chain data available.')
else:
    # Compute median % across assemblies for each biotype × classification
    medians_8 = (
        ic_data.groupby(['biotype', 'classification'])['pct']
        .median()
        .unstack(fill_value=0)
        .reindex(index=BIOTYPE_ORDER, columns=CLASSIFICATION_ORDER, fill_value=0)
    )
    print('Median % across assemblies (8-class):')
    display(medians_8.round(1))

In [ ]:
if not ic_data.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={'width_ratios': [1.2, 1]})

    # ── B1: Full 8-class stacked bars ────────────────────────────────────
    ax = axes[0]
    y_pos = np.arange(len(BIOTYPE_ORDER))
    left = np.zeros(len(BIOTYPE_ORDER))

    for cls in CLASSIFICATION_ORDER:
        vals = medians_8[cls].values
        ax.barh(y_pos, vals, left=left, height=0.65,
                color=CLASS_COLORS[cls], edgecolor='white', linewidth=0.3,
                label=CLASS_LABELS[cls])
        # Label segments > 5%
        for j, (v, l) in enumerate(zip(vals, left)):
            if v >= 5:
                ax.text(l + v/2, j, f'{v:.0f}%',
                        ha='center', va='center', fontsize=7,
                        color='white', fontweight='bold')
        left += vals

    ax.set_yticks(y_pos)
    ax.set_yticklabels([BIOTYPE_LABELS[b] for b in BIOTYPE_ORDER])
    ax.set_xlabel('Percentage of gene pairs (%)')
    ax.set_title('B1. Intron-chain classification (full detail)', fontsize=10, fontweight='bold')
    ax.set_xlim(0, 100)
    ax.invert_yaxis()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Legend below
    handles = [mpatches.Patch(color=CLASS_COLORS[c], label=CLASS_LABELS[c])
               for c in CLASSIFICATION_ORDER]
    ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.12),
              ncol=4, fontsize=7, frameon=False)

    # ── B2: Collapsed 4-group stacked bars ───────────────────────────────
    ax = axes[1]

    # Collapse to 4 groups
    medians_4 = medians_8.copy()
    medians_4.columns = [GROUP_4_MAP[c] for c in medians_4.columns]
    medians_4 = medians_4.T.groupby(level=0).sum().T
    medians_4 = medians_4.reindex(columns=GROUP_4_ORDER, fill_value=0)

    left = np.zeros(len(BIOTYPE_ORDER))
    for grp in GROUP_4_ORDER:
        vals = medians_4[grp].values
        ax.barh(y_pos, vals, left=left, height=0.65,
                color=GROUP_4_COLORS[grp], edgecolor='white', linewidth=0.3,
                label=grp)
        for j, (v, l) in enumerate(zip(vals, left)):
            if v >= 5:
                ax.text(l + v/2, j, f'{v:.0f}%',
                        ha='center', va='center', fontsize=7,
                        color='white', fontweight='bold')
        left += vals

    ax.set_yticks(y_pos)
    ax.set_yticklabels([BIOTYPE_LABELS[b] for b in BIOTYPE_ORDER])
    ax.set_xlabel('Percentage of gene pairs (%)')
    ax.set_title('B2. Grouped (4 categories)', fontsize=10, fontweight='bold')
    ax.set_xlim(0, 100)
    ax.invert_yaxis()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    handles_4 = [mpatches.Patch(color=GROUP_4_COLORS[g], label=g) for g in GROUP_4_ORDER]
    ax.legend(handles=handles_4, loc='upper center', bbox_to_anchor=(0.5, -0.12),
              ncol=4, fontsize=7, frameon=False)

    plt.tight_layout()
    fig.savefig(FIGURE_DIR / 'panel_B_intron_chain_by_biotype.png', dpi=300, bbox_inches='tight')
    fig.savefig(FIGURE_DIR / 'panel_B_intron_chain_by_biotype.pdf', bbox_inches='tight')
    plt.show()
    print('Saved panel_B_intron_chain_by_biotype.{png,pdf}')

---
## Panel C — Jaccard index distribution by biotype

Reconstructed violin/box plots from per-assembly quantiles. Shows the
continuous distribution of exon overlap per gene pair, stratified by biotype.

In [ ]:
if jac_data.empty:
    print('Skipping Panel C — no Jaccard data available.')
else:
    fig, ax = plt.subplots(figsize=(8, 5))

    # Box plots from per-assembly quantiles (using median of medians, etc.)
    bio_stats = []
    for bio in BIOTYPE_ORDER:
        bio_df = jac_data[jac_data['biotype'] == bio]
        if bio_df.empty:
            continue
        bio_stats.append({
            'med':    float(bio_df['median'].median()),
            'q1':     float(bio_df['p25'].median()),
            'q3':     float(bio_df['p75'].median()),
            'whislo': float(bio_df['p5'].median()),
            'whishi': float(bio_df['p95'].median()),
            'fliers': [],
        })

    positions = np.arange(1, len(bio_stats) + 1)
    bp = ax.bxp(bio_stats, positions=positions, widths=0.55,
                patch_artist=True, showfliers=False,
                medianprops=dict(color='black', linewidth=2))

    for i, (patch, bio) in enumerate(zip(bp['boxes'], BIOTYPE_ORDER[:len(bio_stats)])):
        patch.set_facecolor(GROUP_4_COLORS['Exact match'])
        patch.set_alpha(0.6)

    ax.set_xticks(positions)
    ax.set_xticklabels([BIOTYPE_LABELS[b] for b in BIOTYPE_ORDER[:len(bio_stats)]],
                       rotation=25, ha='right')
    ax.set_ylabel('Jaccard index (best-match exon overlap)')
    ax.set_title('C. Exon overlap distribution by biotype', fontsize=10, fontweight='bold')
    ax.set_ylim(-0.05, 1.05)
    ax.axhline(1.0, color='grey', linewidth=0.5, linestyle=':', alpha=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Annotate median values
    for i, stats in enumerate(bio_stats):
        ax.text(i + 1, stats['med'] + 0.03, f"{stats['med']:.2f}",
                ha='center', fontsize=8, fontweight='bold')

    plt.tight_layout()
    fig.savefig(FIGURE_DIR / 'panel_C_jaccard_by_biotype.png', dpi=300, bbox_inches='tight')
    fig.savefig(FIGURE_DIR / 'panel_C_jaccard_by_biotype.pdf', bbox_inches='tight')
    plt.show()
    print('Saved panel_C_jaccard_by_biotype.{png,pdf}')

---
## Panel D — CDS concordance for protein-coding genes

Summary of start codon, stop codon, and reading frame agreement for
protein-coding RBH gene pairs.

In [ ]:
if cds_asm.empty:
    print('Skipping Panel D — no CDS data available.')
else:
    fig, ax = plt.subplots(figsize=(6, 4))

    # Check available columns
    print('CDS per-assembly columns:', list(cds_asm.columns))

    # Try to find full-match percentage column
    pct_col = None
    for candidate in ['pct_Full_Match', 'pct_full_match', 'full_match_pct']:
        if candidate in cds_asm.columns:
            pct_col = candidate
            break

    if pct_col:
        cds_vals = cds_asm[pct_col].dropna()
        cds_med = cds_vals.median()

        # Strip + boxplot
        bp = ax.boxplot(cds_vals, vert=True, widths=0.5, patch_artist=True,
                        boxprops=dict(facecolor='#e76f51', alpha=0.3),
                        medianprops=dict(color='#264653', linewidth=2),
                        flierprops=dict(marker='o', markersize=3, alpha=0.5))
        jitter = np.random.default_rng(42).uniform(-0.15, 0.15, len(cds_vals))
        ax.scatter(np.ones(len(cds_vals)) + jitter, cds_vals, s=6, alpha=0.35,
                   color='#e76f51', edgecolors='none', zorder=3)
        ax.set_ylabel('CDS Full Match (%)')
        ax.set_title(f'D. CDS concordance (protein-coding)\nMedian: {cds_med:.1f}%',
                     fontsize=10, fontweight='bold')
        ax.set_xticks([1])
        ax.set_xticklabels(['All assemblies'])
    else:
        # Fallback: use classification distribution
        if not cds_dist.empty:
            print('Using classification distribution for Panel D')
            display(cds_dist)
            bars = cds_dist.set_index('classification')['median_pct'] if 'median_pct' in cds_dist.columns else None
            if bars is not None:
                bars.plot.bar(ax=ax, color='#e76f51', edgecolor='white')
                ax.set_ylabel('Median % across assemblies')
                ax.set_title('D. CDS classification (protein-coding)',
                             fontsize=10, fontweight='bold')
            else:
                ax.text(0.5, 0.5, 'CDS data format not recognised\nCheck columns',
                        ha='center', va='center', transform=ax.transAxes)
        else:
            ax.text(0.5, 0.5, 'No CDS data', ha='center', va='center',
                    transform=ax.transAxes)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    fig.savefig(FIGURE_DIR / 'panel_D_cds_concordance.png', dpi=300, bbox_inches='tight')
    fig.savefig(FIGURE_DIR / 'panel_D_cds_concordance.pdf', bbox_inches='tight')
    plt.show()
    print('Saved panel_D_cds_concordance.{png,pdf}')

---
## Panel E — GRCh38 divergence categories

Per-assembly violin/box showing: both-agree-reference, both-agree-diverged,
ensembl-specific, cat-specific.

In [ ]:
if div_asm.empty:
    print('Skipping Panel E — no divergence data available.')
else:
    fig, ax = plt.subplots(figsize=(8, 5))

    DIV_COLS = ['pct_both_agree_reference', 'pct_both_agree_diverged',
                'pct_ensembl_specific', 'pct_cat_specific']
    DIV_LABELS = ['Both agree\n(same as ref)', 'Both agree\n(diverged)',
                  'Ensembl-\nspecific', 'CAT-\nspecific']
    DIV_COLORS = ['#2ecc71', '#3498db', '#e67e22', '#9b59b6']

    present_cols = [c for c in DIV_COLS if c in div_asm.columns]
    if present_cols:
        data = [div_asm[c].dropna().values for c in present_cols]
        labels = [DIV_LABELS[DIV_COLS.index(c)] for c in present_cols]
        colors = [DIV_COLORS[DIV_COLS.index(c)] for c in present_cols]

        parts = ax.violinplot(data, showmedians=True)
        for i, (pc, col) in enumerate(zip(parts['bodies'], colors)):
            pc.set_facecolor(col)
            pc.set_alpha(0.6)

        ax.set_xticks(range(1, len(labels) + 1))
        ax.set_xticklabels(labels, fontsize=9)
        ax.set_ylabel('Percentage per assembly (%)')
        ax.set_title('E. GRCh38 divergence categories', fontsize=10, fontweight='bold')

        # Annotate medians
        for i, d in enumerate(data):
            m = np.median(d)
            ax.text(i + 1, m + 1, f'{m:.1f}%', ha='center', fontsize=8, fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'Divergence columns not found\n' + str(div_asm.columns.tolist()[:5]),
                ha='center', va='center', transform=ax.transAxes, fontsize=8)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    fig.savefig(FIGURE_DIR / 'panel_E_divergence.png', dpi=300, bbox_inches='tight')
    fig.savefig(FIGURE_DIR / 'panel_E_divergence.pdf', bbox_inches='tight')
    plt.show()
    print('Saved panel_E_divergence.{png,pdf}')

---
## Composite figure draft

Assemble selected panels into a single multi-panel figure.
Edit the cell below to choose which Panel A style to use.

In [ ]:
# ── Configuration: Choose Panel A style ──────────────────────────────────
PANEL_A_STYLE = 'strip_box'  # 'strip_box', 'horiz_bar', or 'violin'
USE_8_CLASS = True           # True for full detail, False for 4-group summary


def draw_panel_A(ax, style='strip_box'):
    """Draw gene-level agreement panel."""
    vals = gene_pres['pct'].dropna()
    med = vals.median()

    if style == 'strip_box':
        ax.boxplot(vals, vert=True, widths=0.5, patch_artist=True,
                   boxprops=dict(facecolor='#2a9d8f', alpha=0.3),
                   medianprops=dict(color='#264653', linewidth=2),
                   whiskerprops=dict(color='#264653'),
                   capprops=dict(color='#264653'),
                   flierprops=dict(marker='o', markersize=2, alpha=0.4))
        jitter = np.random.default_rng(42).uniform(-0.15, 0.15, len(vals))
        ax.scatter(np.ones(len(vals)) + jitter, vals, s=4, alpha=0.3,
                   color='#2a9d8f', edgecolors='none', zorder=3)
        ax.set_xticks([1])
        ax.set_xticklabels([''])
        ax.set_ylabel('Gene loci detected\nby both methods (%)')
        ax.set_ylim(max(vals.min() - 1, 90), 100)

    elif style == 'horiz_bar':
        ax.barh(0, med, height=0.5, color='#2a9d8f', alpha=0.85, edgecolor='white')
        ax.text(med + 0.1, 0, f'{med:.1f}%', va='center', fontsize=8, fontweight='bold')
        jitter_y = np.random.default_rng(42).uniform(-0.18, 0.18, len(vals))
        ax.scatter(vals, jitter_y, s=4, alpha=0.35, color='#264653', edgecolors='none', zorder=3)
        ax.set_xlabel('Gene loci detected by both methods (%)')
        ax.set_yticks([])
        ax.set_xlim(max(vals.min() - 1, 90), 101)

    elif style == 'violin':
        parts = ax.violinplot(vals, vert=True, showmedians=True)
        for pc in parts['bodies']:
            pc.set_facecolor('#2a9d8f')
            pc.set_alpha(0.5)
        parts['cmedians'].set_color('#264653')
        parts['cmedians'].set_linewidth(2)
        ax.set_xticks([1])
        ax.set_xticklabels([''])
        ax.set_ylabel('Gene loci detected\nby both methods (%)')
        ax.set_ylim(max(vals.min() - 1, 90), 100)

    ax.set_title(f'Median: {med:.1f}%', fontsize=8, style='italic')


def draw_panel_B(ax, use_8_class=True):
    """Draw intron-chain concordance by biotype."""
    if ic_data.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        return

    y_pos = np.arange(len(BIOTYPE_ORDER))
    left = np.zeros(len(BIOTYPE_ORDER))

    if use_8_class:
        order = CLASSIFICATION_ORDER
        colors = CLASS_COLORS
        labels = CLASS_LABELS
        data = medians_8
    else:
        order = GROUP_4_ORDER
        colors = GROUP_4_COLORS
        labels = {g: g for g in GROUP_4_ORDER}
        data = medians_4

    for cls in order:
        vals = data[cls].values
        ax.barh(y_pos, vals, left=left, height=0.65,
                color=colors[cls], edgecolor='white', linewidth=0.3,
                label=labels[cls])
        for j, (v, l) in enumerate(zip(vals, left)):
            if v >= 6:
                ax.text(l + v/2, j, f'{v:.0f}%',
                        ha='center', va='center', fontsize=6,
                        color='white', fontweight='bold')
        left += vals

    ax.set_yticks(y_pos)
    ax.set_yticklabels([BIOTYPE_LABELS[b] for b in BIOTYPE_ORDER])
    ax.set_xlabel('Percentage of gene pairs (%)')
    ax.set_xlim(0, 100)
    ax.invert_yaxis()


def draw_panel_C(ax):
    """Draw Jaccard index distribution."""
    if jac_data.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        return

    bio_stats = []
    bio_labels = []
    for bio in BIOTYPE_ORDER:
        bio_df = jac_data[jac_data['biotype'] == bio]
        if bio_df.empty:
            continue
        bio_stats.append({
            'med': float(bio_df['median'].median()),
            'q1': float(bio_df['p25'].median()),
            'q3': float(bio_df['p75'].median()),
            'whislo': float(bio_df['p5'].median()),
            'whishi': float(bio_df['p95'].median()),
            'fliers': [],
        })
        bio_labels.append(BIOTYPE_LABELS[bio])

    positions = np.arange(1, len(bio_stats) + 1)
    bp = ax.bxp(bio_stats, positions=positions, widths=0.5,
                patch_artist=True, showfliers=False,
                medianprops=dict(color='black', linewidth=1.5))
    for patch in bp['boxes']:
        patch.set_facecolor('#2a9d8f')
        patch.set_alpha(0.5)

    ax.set_xticks(positions)
    ax.set_xticklabels(bio_labels, rotation=30, ha='right')
    ax.set_ylabel('Jaccard index')
    ax.set_ylim(-0.05, 1.05)
    ax.axhline(1.0, color='grey', linewidth=0.5, linestyle=':', alpha=0.4)

    for i, stats in enumerate(bio_stats):
        ax.text(i + 1, stats['med'] + 0.03, f"{stats['med']:.2f}",
                ha='center', fontsize=7, fontweight='bold')


def draw_panel_D(ax):
    """Draw CDS concordance summary."""
    if cds_asm.empty:
        ax.text(0.5, 0.5, 'No CDS data', ha='center', va='center', transform=ax.transAxes)
        return

    pct_col = None
    for candidate in ['pct_Full_Match', 'pct_full_match', 'full_match_pct']:
        if candidate in cds_asm.columns:
            pct_col = candidate
            break

    if pct_col:
        cds_vals = cds_asm[pct_col].dropna()
        cds_med = cds_vals.median()
        bp = ax.boxplot(cds_vals, vert=True, widths=0.5, patch_artist=True,
                        boxprops=dict(facecolor='#e76f51', alpha=0.3),
                        medianprops=dict(color='#264653', linewidth=2),
                        flierprops=dict(marker='o', markersize=2, alpha=0.4))
        jitter = np.random.default_rng(42).uniform(-0.15, 0.15, len(cds_vals))
        ax.scatter(np.ones(len(cds_vals)) + jitter, cds_vals, s=4, alpha=0.3,
                   color='#e76f51', edgecolors='none', zorder=3)
        ax.set_ylabel('CDS Full Match (%)')
        ax.set_xticks([1])
        ax.set_xticklabels([''])
        ax.set_title(f'Median: {cds_med:.1f}%', fontsize=8, style='italic')
    else:
        ax.text(0.5, 0.5, f'Column not found\n{cds_asm.columns.tolist()[:4]}',
                ha='center', va='center', transform=ax.transAxes, fontsize=7)


# ── Assemble composite figure ────────────────────────────────────────────
fig = plt.figure(figsize=(14, 10))
gs = GridSpec(2, 3, figure=fig,
              height_ratios=[1, 1.2],
              width_ratios=[0.8, 2, 1],
              hspace=0.35, wspace=0.35)

# Top row: A (small), B (wide), C (medium)
ax_A = fig.add_subplot(gs[0, 0])
ax_B = fig.add_subplot(gs[0, 1])
ax_C = fig.add_subplot(gs[0, 2])

# Bottom row: D (small), E (spanning)
ax_D = fig.add_subplot(gs[1, 0])
ax_E = fig.add_subplot(gs[1, 1:])

# Draw panels
draw_panel_A(ax_A, style=PANEL_A_STYLE)
draw_panel_B(ax_B, use_8_class=USE_8_CLASS)
draw_panel_C(ax_C)
draw_panel_D(ax_D)

# Panel E: divergence
if not div_asm.empty:
    DIV_COLS = ['pct_both_agree_reference', 'pct_both_agree_diverged',
                'pct_ensembl_specific', 'pct_cat_specific']
    DIV_LABELS_SHORT = ['Both agree\n(ref)', 'Both agree\n(diverged)',
                        'Ensembl-\nspecific', 'CAT-\nspecific']
    DIV_COLORS = ['#2ecc71', '#3498db', '#e67e22', '#9b59b6']
    present_cols = [c for c in DIV_COLS if c in div_asm.columns]
    if present_cols:
        data = [div_asm[c].dropna().values for c in present_cols]
        parts = ax_E.violinplot(data, showmedians=True)
        for i, (pc, col) in enumerate(zip(parts['bodies'],
                                           [DIV_COLORS[DIV_COLS.index(c)] for c in present_cols])):
            pc.set_facecolor(col)
            pc.set_alpha(0.6)
        ax_E.set_xticks(range(1, len(present_cols) + 1))
        ax_E.set_xticklabels([DIV_LABELS_SHORT[DIV_COLS.index(c)] for c in present_cols], fontsize=8)
        ax_E.set_ylabel('% per assembly')
        for i, d in enumerate(data):
            m = np.median(d)
            ax_E.text(i + 1, m + 1, f'{m:.1f}%', ha='center', fontsize=7, fontweight='bold')
else:
    ax_E.text(0.5, 0.5, 'No divergence data', ha='center', va='center', transform=ax_E.transAxes)

# Panel labels
for ax, label in [(ax_A, 'A'), (ax_B, 'B'), (ax_C, 'C'), (ax_D, 'D'), (ax_E, 'E')]:
    ax.text(-0.15, 1.08, label, transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='top')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Panel B legend (below the axes to avoid clutter)
if not ic_data.empty:
    if USE_8_CLASS:
        handles = [mpatches.Patch(color=CLASS_COLORS[c], label=CLASS_LABELS[c])
                   for c in CLASSIFICATION_ORDER]
    else:
        handles = [mpatches.Patch(color=GROUP_4_COLORS[g], label=g) for g in GROUP_4_ORDER]
    fig.legend(handles=handles, loc='lower center', ncol=4 if not USE_8_CLASS else 4,
              fontsize=7, frameon=False, bbox_to_anchor=(0.5, -0.02))

fig.savefig(FIGURE_DIR / 'figure_main_concordance_v2.png', dpi=300, bbox_inches='tight')
fig.savefig(FIGURE_DIR / 'figure_main_concordance_v2.pdf', bbox_inches='tight')
plt.show()
print('Saved figure_main_concordance_v2.{png,pdf}')

---
## Summary statistics for manuscript text

In [ ]:
print('=== Key numbers for manuscript text ===')
print()

# Gene presence
vals = gene_pres['pct'].dropna()
print(f'Gene-level agreement:')
print(f'  Median: {vals.median():.1f}%  (IQR: {vals.quantile(0.25):.1f}–{vals.quantile(0.75):.1f}%)')
print(f'  Range:  {vals.min():.1f}–{vals.max():.1f}%')
print(f'  N assemblies: {len(vals)}')
print()

# Transcript concordance by biotype (if available)
if not ic_data.empty:
    print('Transcript concordance by biotype (median % across assemblies):')
    for bio in BIOTYPE_ORDER:
        if bio in medians_8.index:
            exact = medians_8.loc[bio, 'Exact_Match']
            intron = medians_8.loc[bio, 'Intron_Match']
            no_match = medians_8.loc[bio, 'No_Match']
            same_chain = exact + intron + medians_8.loc[bio, 'Intron_Subset'] + medians_8.loc[bio, 'Intron_Superset']
            print(f'  {BIOTYPE_LABELS[bio]:20s}: '
                  f'Exact={exact:.1f}%, Same intron chain={same_chain:.1f}%, No match={no_match:.1f}%')
    print()

# Jaccard
if not jac_data.empty:
    print('Jaccard index (median of medians across assemblies):')
    for bio in BIOTYPE_ORDER:
        bio_df = jac_data[jac_data['biotype'] == bio]
        if not bio_df.empty:
            print(f'  {BIOTYPE_LABELS[bio]:20s}: median={bio_df["median"].median():.3f}, '
                  f'IQR={bio_df["p25"].median():.3f}–{bio_df["p75"].median():.3f}')
    print()

print('Done.')